<a href="https://colab.research.google.com/github/kushim2005/omniproject/blob/main/Citation_Source_Backend_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q sentence-transformers faiss-cpu pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.9 MB/s eta 0:00:00


In [3]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Project folder"

FAISS_PATH = os.path.join(
    PROJECT_PATH,
    "faiss_index.bin"
)

METADATA_PATH = os.path.join(
    PROJECT_PATH,
    "metadata.pkl"
)

print("Project path:", PROJECT_PATH)
print("FAISS exists:", os.path.exists(FAISS_PATH))
print("Metadata exists:", os.path.exists(METADATA_PATH))

Project path: /content/drive/MyDrive/Project folder
FAISS exists: True
Metadata exists: True


In [4]:
import glob

pdf_files = glob.glob(
    os.path.join(PROJECT_PATH, "*.pdf")
)

print("PDF files found:")

for pdf in pdf_files:
    print(pdf)

PDF files found:
/content/drive/MyDrive/Project folder/ML.pdf


In [5]:
import faiss
import pickle

index = faiss.read_index(FAISS_PATH)

with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata records:", len(metadata))

FAISS vectors: 488
Metadata records: 488


In [6]:
print(metadata[0])

{'id': 0, 'text': 'MACHINE LEARNING  \n[R17A0534] \nLECTURE NOTES \n \nB.TECH IV YEAR – I SEM(R17) \n(2020-21) \n \n \n \n \n \n \nDEPARTMENT OF \nCOMPUTER SCIENCE AND ENGINEERING \nMALLA REDDY COLLEGE OF ENGINEERING & \nTECHNOLOGY \n(Autonomous Institution – UGC, Govt. of India) \nRecognized under 2(f) and 12 (B) of UGC ACT 1956 \n(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) \nMaisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State'}


In [10]:
import fitz

doc = fitz.open(pdf)

page_texts = []

for page_number, page in enumerate(doc, start=1):

    page_text = page.get_text()

    page_texts.append({
        "page": page_number,
        "text": page_text
    })

doc.close()

print("Total PDF pages:", len(page_texts))

Total PDF pages: 120


In [11]:
full_text = ""

for page in page_texts:
    full_text += page["text"]

print("Total characters:", len(full_text))

Total characters: 243696


In [12]:
chunk_size = 500

expected_chunks = [
    full_text[i:i + chunk_size]
    for i in range(0, len(full_text), chunk_size)
]

print("Reconstructed chunks:", len(expected_chunks))
print("Existing metadata:", len(metadata))

Reconstructed chunks: 488
Existing metadata: 488


In [14]:
enhanced_metadata = []

current_position = 0

for i, item in enumerate(metadata):

    chunk_text = item["text"]

    start_position = current_position
    end_position = current_position + len(chunk_text)

    # Determine which PDF page contains the beginning of the chunk
    cumulative_position = 0
    page_number = None

    for page_info in page_texts:

        page_start = cumulative_position
        page_end = cumulative_position + len(page_info["text"])

        if page_start <= start_position < page_end:
            page_number = page_info["page"]
            break

        cumulative_position = page_end

    if page_number is None:
        page_number = 1

    enhanced_metadata.append({
        "chunk_id": f"chunk_{i}",
        "source": os.path.basename(pdf),
        "page": page_number,
        "text": chunk_text
    })

    current_position = end_position

print("Enhanced metadata created:", len(enhanced_metadata))

Enhanced metadata created: 488


In [15]:
for item in enhanced_metadata[:3]:

    print("=" * 70)

    print("Chunk ID:", item["chunk_id"])
    print("Source:", item["source"])
    print("Page:", item["page"])
    print("Text:", item["text"][:300])

Chunk ID: chunk_0
Source: ML.pdf
Page: 1
Text: MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Af
Chunk ID: chunk_1
Source: ML.pdf
Page: 1
Text: , India 
 
IV Year B. Tech. CSE –II Sem   
 
 
 
 
 
                L   T/P/D   C  
  4   1/- / -   3  
(R17A0534) Machine Learning 
Objectives:  
 Acquire theoretical Knowledge on setting hypothesis for pattern recognition. 
 Apply suitable machine learning techniques for data handling and to ga
Chunk ID: chunk_2
Source: ML.pdf
Page: 2
Text:  Learning , Learning Models , Geometric Models, Probabilistic 
Models, Logic Models, Grouping and Grading, Designing a Learning System, Types of 
Learning, Supervised, Unsupervised, Reinforcement, Perspectives and Issues, Version Spaces, 
PAC Learning, VC D

In [16]:
CITATION_METADATA_PATH = os.path.join(
    PROJECT_PATH,
    "citation_metadata.pkl"
)

with open(CITATION_METADATA_PATH, "wb") as f:
    pickle.dump(enhanced_metadata, f)

print("Saved:", CITATION_METADATA_PATH)

Saved: /content/drive/MyDrive/Project folder/citation_metadata.pkl


In [17]:
import json

CITATION_JSON_PATH = os.path.join(
    PROJECT_PATH,
    "citation_metadata.json"
)

with open(CITATION_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        enhanced_metadata,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved:", CITATION_JSON_PATH)

Saved: /content/drive/MyDrive/Project folder/citation_metadata.json


In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [19]:
import numpy as np

def search_with_citations(query, k=3):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search existing FAISS index
    distances, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        item = enhanced_metadata[idx]

        distance = float(distances[0][rank])

        # Convert FAISS L2 distance into an easy-to-read relevance score.
        # Higher score = more relevant.
        relevance_score = 1 / (1 + distance)

        result = {
            "rank": rank + 1,
            "source": item["source"],
            "page": item["page"],
            "chunk_id": item["chunk_id"],
            "text": item["text"],
            "distance": distance,
            "relevance_score": round(
                relevance_score,
                4
            )
        }

        results.append(result)

    return results

In [20]:
query = "What is supervised learning?"

results = search_with_citations(
    query,
    k=3
)

for result in results:

    print("=" * 80)

    print("Rank:", result["rank"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print("Distance:", result["distance"])
    print("Relevance Score:", result["relevance_score"])

    print("\nRetrieved Text:")
    print(result["text"])

Rank: 1
Source: ML.pdf
Page: 17
Chunk ID: chunk_69
Distance: 0.47794026136398315
Relevance Score: 0.6766

Retrieved Text:
puts. This is also called 
learning from exemplars. Supervised learning is the machine learning task of learning a function that 
maps an input to an output based on example input-output pairs. 
 
In supervised learning, each example in the training set is a pair consisting of an input object 
(typically a vector) and an output value. A supervised learning algorithm analyzes the training data and 
produces a function, which can be used for mapping new examples. In the optimal case, the function 
Rank: 2
Source: ML.pdf
Page: 61
Chunk ID: chunk_243
Distance: 0.7442344427108765
Relevance Score: 0.5733

Retrieved Text:
ed learning is to find the hidden patterns and 
useful insights from the unknown dataset. 
Supervised learning needs supervision to train the model. 
Unsupervised learning does not need any supervision to train the 
model. 
Supervised learning can be cate

In [21]:
def create_citations(results):

    citations = []

    for result in results:

        citation = {
            "citation_id": f"[{result['rank']}]",
            "source": result["source"],
            "page": result["page"],
            "chunk_id": result["chunk_id"],
            "relevance_score": result["relevance_score"],
            "text": result["text"]
        }

        citations.append(citation)

    return citations

In [22]:
citations = create_citations(results)

for citation in citations:

    print("=" * 80)

    print("Citation:", citation["citation_id"])
    print("Source:", citation["source"])
    print("Page:", citation["page"])
    print("Chunk:", citation["chunk_id"])
    print("Score:", citation["relevance_score"])

Citation: [1]
Source: ML.pdf
Page: 17
Chunk: chunk_69
Score: 0.6766
Citation: [2]
Source: ML.pdf
Page: 61
Chunk: chunk_243
Score: 0.5733
Citation: [3]
Source: ML.pdf
Page: 17
Chunk: chunk_70
Score: 0.5629


In [23]:
def build_context(results):

    context_parts = []

    for result in results:

        context_parts.append(
            f"""
Source: {result['source']}
Page: {result['page']}
Chunk ID: {result['chunk_id']}

Content:
{result['text']}
"""
        )

    return "\n".join(context_parts)

In [24]:
context = build_context(results)

print(context)


Source: ML.pdf
Page: 17
Chunk ID: chunk_69

Content:
puts. This is also called 
learning from exemplars. Supervised learning is the machine learning task of learning a function that 
maps an input to an output based on example input-output pairs. 
 
In supervised learning, each example in the training set is a pair consisting of an input object 
(typically a vector) and an output value. A supervised learning algorithm analyzes the training data and 
produces a function, which can be used for mapping new examples. In the optimal case, the function 


Source: ML.pdf
Page: 61
Chunk ID: chunk_243

Content:
ed learning is to find the hidden patterns and 
useful insights from the unknown dataset. 
Supervised learning needs supervision to train the model. 
Unsupervised learning does not need any supervision to train the 
model. 
Supervised learning can be categorized 
in Classification and Regression problems. 
Unsupervised Learning can be classified 
in Clustering and Associations problems.

In [25]:
def create_rag_response(answer, results):

    citations = create_citations(results)

    response = {
        "answer": answer,
        "citations": citations
    }

    return response

In [26]:
answer = (
    "Supervised learning uses known examples to learn "
    "the relationship between input and output."
)

final_response = create_rag_response(
    answer,
    results
)

print(
    json.dumps(
        final_response,
        indent=4,
        ensure_ascii=False
    )
)

{
    "answer": "Supervised learning uses known examples to learn the relationship between input and output.",
    "citations": [
        {
            "citation_id": "[1]",
            "source": "ML.pdf",
            "page": 17,
            "chunk_id": "chunk_69",
            "relevance_score": 0.6766,
            "text": "puts. This is also called \nlearning from exemplars. Supervised learning is the machine learning task of learning a function that \nmaps an input to an output based on example input-output pairs. \n \nIn supervised learning, each example in the training set is a pair consisting of an input object \n(typically a vector) and an output value. A supervised learning algorithm analyzes the training data and \nproduces a function, which can be used for mapping new examples. In the optimal case, the function "
        },
        {
            "citation_id": "[2]",
            "source": "ML.pdf",
            "page": 61,
            "chunk_id": "chunk_243",
            "relev

In [27]:
test_queries = [
    "What is supervised learning?",
    "What is deep learning?",
    "What is reinforcement learning?",
    "What is k-nearest neighbours?",
    "What is an artificial neural network?"
]

for query in test_queries:

    print("\n")
    print("#" * 80)
    print("QUERY:", query)
    print("#" * 80)

    results = search_with_citations(
        query,
        k=3
    )

    for result in results:

        print(
            f"[{result['rank']}] "
            f"{result['source']} | "
            f"Page {result['page']} | "
            f"{result['chunk_id']} | "
            f"Score {result['relevance_score']}"
        )

        print(
            result["text"][:250].replace("\n", " ")
        )

        print()



################################################################################
QUERY: What is supervised learning?
################################################################################
[1] ML.pdf | Page 17 | chunk_69 | Score 0.6766
puts. This is also called  learning from exemplars. Supervised learning is the machine learning task of learning a function that  maps an input to an output based on example input-output pairs.    In supervised learning, each example in the training 

[2] ML.pdf | Page 61 | chunk_243 | Score 0.5733
ed learning is to find the hidden patterns and  useful insights from the unknown dataset.  Supervised learning needs supervision to train the model.  Unsupervised learning does not need any supervision to train the  model.  Supervised learning can be

[3] ML.pdf | Page 17 | chunk_70 | Score 0.5629
 will correctly determine the class labels for unseen instances. Both classification and regression  13    problems are supervised learning problems. A wi

In [28]:
def format_citations(results):

    formatted = []

    for result in results:

        formatted.append({
            "citation": f"[{result['rank']}]",
            "source": result["source"],
            "page": result["page"],
            "chunk_id": result["chunk_id"],
            "score": result["relevance_score"]
        })

    return formatted

In [29]:
results = search_with_citations(
    "What is deep learning?",
    k=3
)

formatted = format_citations(results)

print(
    json.dumps(
        formatted,
        indent=4
    )
)

[
    {
        "citation": "[1]",
        "source": "ML.pdf",
        "page": 6,
        "chunk_id": "chunk_11",
        "score": 0.5081
    },
    {
        "citation": "[2]",
        "source": "ML.pdf",
        "page": 6,
        "chunk_id": "chunk_12",
        "score": 0.5013
    },
    {
        "citation": "[3]",
        "source": "ML.pdf",
        "page": 17,
        "chunk_id": "chunk_69",
        "score": 0.4996
    }
]


In [30]:
test_output = {
    "query": "What is deep learning?",
    "results": results,
    "citations": format_citations(results)
}

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "citation_test_output.json"
)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:

    json.dump(
        test_output,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved:", OUTPUT_PATH)

Saved: /content/drive/MyDrive/Project folder/citation_test_output.json


In [31]:
def citation_backend(query, top_k=3):

    # Step 1: Search existing vector database
    results = search_with_citations(
        query,
        k=top_k
    )

    # Step 2: No results
    if not results:

        return {
            "query": query,
            "answer": "No relevant information found.",
            "citations": []
        }

    # Step 3: Build citations
    citations = create_citations(results)

    # Step 4: Build context
    context = build_context(results)

    # Step 5: Return backend response
    return {
        "query": query,
        "context": context,
        "citations": citations
    }

In [32]:
result = citation_backend(
    "What is supervised learning?",
    top_k=3
)

print(
    json.dumps(
        result,
        indent=4,
        ensure_ascii=False
    )
)

{
    "query": "What is supervised learning?",
    "context": "\nSource: ML.pdf\nPage: 17\nChunk ID: chunk_69\n\nContent:\nputs. This is also called \nlearning from exemplars. Supervised learning is the machine learning task of learning a function that \nmaps an input to an output based on example input-output pairs. \n \nIn supervised learning, each example in the training set is a pair consisting of an input object \n(typically a vector) and an output value. A supervised learning algorithm analyzes the training data and \nproduces a function, which can be used for mapping new examples. In the optimal case, the function \n\n\nSource: ML.pdf\nPage: 61\nChunk ID: chunk_243\n\nContent:\ned learning is to find the hidden patterns and \nuseful insights from the unknown dataset. \nSupervised learning needs supervision to train the model. \nUnsupervised learning does not need any supervision to train the \nmodel. \nSupervised learning can be categorized \nin Classification and Regression pro